# Implementation of SimCLR with training the encoder and after the classification head

**The encoder can be pretrained using an unsupervised way. It is after trained using the SimCLR algorithm and the classification head is trained by supervised learning**

In [ ]:
import os

os.environ["CUDA_VISIBLE_DEVICES"] = "-1"  # disable GPU devices
os.environ["TFDS_DATA_DIR"] = os.path.expanduser("~/tensorflow_datasets")  # default location of tfds database
os.environ["KERAS_BACKEND"] = "tensorflow"

import keras
from keras import layers, models, regularizers
from keras.callbacks import EarlyStopping, ReduceLROnPlateau

import tensorflow as tf
import tensorflow_datasets as tfds

from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
from sklearn.manifold import TSNE

# Turn off logging for TF
import logging
logging.disable(logging.WARNING)
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"
tf.get_logger().setLevel(logging.ERROR)

from dpmhm.datasets import preprocessing, feature, utils, transformer, query_parameters

In [ ]:
nb_elem_dirg=10000
nb_elem_paderborn= 10000
ds_train_size = nb_elem_dirg+nb_elem_paderborn

batch_size = 64
n_embedding  = 256 
kernel_size = (3,3) 
tau = 0.1
projection_dim = 128 

# Preprocessing on data

Load registered datasets (if not registered, run creation_metadataset.ipynb)

In [ ]:
outdir = Path('/volatile/home/bm279471/tmp/meta_dataset')
# outdir = Path('/volatile/home/bm279471/tmp/meta_dataset_full_bandwidth')
# outdir = Path('/volatile/home/bm279471/tmp/few_shot_cwru')
os.makedirs(outdir, exist_ok=True)

In [ ]:
import json

ds_train = tf.data.Dataset.load(str(outdir/'ds_train'))
ds_val = tf.data.Dataset.load(str(outdir/'ds_val'))
ds_train_ft = tf.data.Dataset.load(str(outdir/'ds_train_ft'))
ds_val_ft = tf.data.Dataset.load(str(outdir/'ds_val_ft'))
ds_test_ft = tf.data.Dataset.load(str(outdir/'ds_test_ft'))

with open(outdir/'lb1.json', 'r') as fp:
    lb1 = list(json.load(fp))
with open(outdir/'lb2.json', 'r') as fp:
    lb2 = list(json.load(fp))
with open(outdir/'lb3.json', 'r') as fp:
    lb3 = list(json.load(fp))

ds_train_clr = ds_train.map(lambda x,l:(x,x)).shuffle(ds_train_size, reshuffle_each_iteration=False).cache().batch(batch_size).prefetch(tf.data.AUTOTUNE)
ds_val_clr = ds_val.cache().batch(batch_size,drop_remainder=True)

ds_train_class = ds_train.shuffle(ds_train_size, reshuffle_each_iteration=False).cache().batch(batch_size).prefetch(tf.data.AUTOTUNE)
ds_val_class = ds_val.cache().batch(batch_size,drop_remainder=True)

ds_test_size = utils.get_dataset_size(ds_train_ft)+utils.get_dataset_size(ds_val_ft)+utils.get_dataset_size(ds_test_ft)

ds_train_ft= ds_train_ft.shuffle(ds_test_size, reshuffle_each_iteration=False).cache().batch(batch_size,drop_remainder=True).prefetch(tf.data.AUTOTUNE)
ds_val_ft = ds_val_ft.cache().batch(batch_size,drop_remainder=True)
ds_test_ft=ds_test_ft.cache().batch(1)

In [ ]:
ds_test_size = utils.get_dataset_size(ds_train_ft)+utils.get_dataset_size(ds_val_ft)+utils.get_dataset_size(ds_test_ft)
eles = list(ds_train_clr.take(1).as_numpy_iterator())
input_shape = eles[0][0][0].shape

# With an encoder from scratch

**Autoencoder**

Create the autoencoder

In [ ]:
@tf.keras.utils.register_keras_serializable()
class Encoder(models.Model):
    """Convolution Auto-Encoder stacks.

    Notes
    -----
    Shape (H,W) of the input tensor must be power of 2.
    """
    def __init__(self, input_shape, n_embedding, kernel_size):
        super(Encoder, self).__init__()
        activation = 'relu'
        padding = 'same'
        strides = (2, 2)
        pool_size = (2, 2)
        a_reg = 0.

        # Encoder
        input_enc = layers.Input(shape=input_shape, name='input_enc')
        x = layers.Conv2D(32, kernel_size=kernel_size, activation=activation, padding=padding, name='conv1_enc')(input_enc)
        x = layers.MaxPooling2D(pool_size=pool_size, strides=strides, name='pool1_enc')(x)
        x = layers.BatchNormalization(name='bn1_enc')(x)

        x = layers.Conv2D(64, kernel_size=kernel_size, activation=activation, padding=padding, name='conv2_enc')(x)
        x = layers.MaxPooling2D(pool_size=pool_size, strides=strides, name='pool2_enc')(x)
        x = layers.BatchNormalization(name='bn2_enc')(x)

        x = layers.Conv2D(128, kernel_size=kernel_size, activation=activation, padding=padding, name='conv3_enc')(x)
        x = layers.MaxPooling2D(pool_size=pool_size, strides=strides, name='pool3_enc')(x)
        x = layers.BatchNormalization(name='bn3_enc')(x)

        x = layers.Flatten(name='flatten')(x)
        if a_reg > 0:
            x = layers.Dense(n_embedding, activation=activation, activity_regularizer=regularizers.L1(a_reg), name='fc1_enc')(x)
        else:
            x = layers.Dense(n_embedding, activation=activation, name='fc1_enc')(x)

        self.encoder = models.Model(input_enc, x, name='encoder')

    def call(self, x, training=False):
        return self.encoder(x, training=training)

In [ ]:
encoder = Encoder(input_shape,n_embedding,kernel_size)

**Contrastive learning on the encoder**

Contrastive loss function

In [ ]:
def contrastive_loss_fn(z_i, z_j, tau=0.5):
    z_i = tf.math.l2_normalize(z_i, axis=1)
    z_j = tf.math.l2_normalize(z_j, axis=1)

    # Compute the similarity matrix
    similarity_matrix = tf.matmul(z_i, z_j, transpose_b=True) / tau

    # Compute the positive similarity
    positive_similarity = tf.linalg.diag_part(similarity_matrix)

    # Compute the negative similarity
    negative_similarity = tf.linalg.set_diag(similarity_matrix, tf.zeros_like(tf.linalg.diag_part(similarity_matrix)))

    # Compute the numerator of the loss function
    numerator = tf.exp(positive_similarity)

    # Compute the denominator of the loss function
    denominator = tf.reduce_sum(tf.exp(negative_similarity), axis=1)

    # Compute the loss function
    loss = -tf.reduce_mean(tf.math.log(numerator / denominator))

    return loss

Create the SimCLR model

In [ ]:
tf.config.experimental_run_functions_eagerly(True)

# Define the contrastive model with model-subclassing
class SimCLRModel(keras.Model):
    def __init__(self, encoder):
        super().__init__()

        self.tau = tau

        self.contrastive_augmenter = keras.Sequential([
            layers.RandomFlip("horizontal_and_vertical"),
            layers.RandomZoom(0.2),
            layers.RandomTranslation(height_factor=0.2, width_factor=0.2),
        ], name='Data_augmentation')
        
        self.encoder = encoder

        self.projection_head = keras.Sequential([
                layers.Dense(256, activation='relu'),
                layers.BatchNormalization(),
                layers.Dense(128, activation='relu'),
                layers.BatchNormalization(),
                layers.Dense(projection_dim),
            ], name='Projection_head')

    def compile(self, contrastive_optimizer, **kwargs):
        super().compile(**kwargs)

        self.contrastive_optimizer = contrastive_optimizer

        self.contrastive_loss_tracker = keras.metrics.Mean(name="c_loss")

    @property
    def metrics(self):
        return [
            self.contrastive_loss_tracker
        ]

    def train_step(self, data):
        train_image, _ = data
        
        # Each set of unlabeled images is augmented separately
        augmented_image_1 = self.contrastive_augmenter(train_image, training=True)
        augmented_image_2 = self.contrastive_augmenter(train_image, training=True)
        
        with tf.GradientTape() as tape:
            # Extract features and compute projections for the first set of images
            features_1 = self.encoder(augmented_image_1, training=True)
            projections_1 = self.projection_head(features_1, training=True)
            
            # Extract features and compute projections for the second set of images
            features_2 = self.encoder(augmented_image_2, training=True)
            projections_2 = self.projection_head(features_2, training=True)
            
            # Compute contrastive loss
            contrastive_loss = contrastive_loss_fn(projections_1, projections_2, self.tau) +contrastive_loss_fn(projections_2, projections_1, self.tau)
        
        # Compute gradients and apply updates for the contrastive loss
        gradients = tape.gradient(
            contrastive_loss,
            self.encoder.trainable_weights + self.projection_head.trainable_weights,
        )
        self.contrastive_optimizer.apply_gradients(
            zip(
                gradients,
                self.encoder.trainable_weights + self.projection_head.trainable_weights,
            )
        )
        self.contrastive_loss_tracker.update_state(contrastive_loss)
        return {m.name: m.result() for m in self.metrics}

    def test_step(self, data):
        test_image, _ = data

        # Compute contrastive loss on the validation set
        augmented_image_1 = self.contrastive_augmenter(test_image, training=True)
        augmented_image_2 = self.contrastive_augmenter(test_image, training=True)

        features_1 = self.encoder(augmented_image_1, training=False)
        features_2 = self.encoder(augmented_image_2, training=False)

        projections_1 = self.projection_head(features_1, training=False)
        projections_2 = self.projection_head(features_2, training=False)
        
        contrastive_loss = contrastive_loss_fn(projections_1, projections_2, self.tau) + contrastive_loss_fn(projections_2, projections_1, self.tau)
        self.contrastive_loss_tracker.update_state(contrastive_loss)
        return {m.name: m.result() for m in self.metrics}

Compile and train SimCLR model

In [ ]:
simclr_model = SimCLRModel(encoder)

simclr_model.compile(
    contrastive_optimizer=keras.optimizers.Adam(1e-4),
)

early_stopping = EarlyStopping(
    monitor='val_c_loss',  
    patience=4,       
    restore_best_weights=True, 
    mode='min'
)

reduce_lr = ReduceLROnPlateau(
    monitor='val_c_loss', 
    factor=0.1,         
    patience=2, 
    mode='min'    
)

simclr_history = simclr_model.fit(
    ds_train_clr.repeat(), 
    epochs=30, #45
    validation_data=ds_val_clr, 
    steps_per_epoch=int((0.8*ds_train_size) // batch_size),
    callbacks=[early_stopping, reduce_lr]
)

simclr_model.summary()

In [ ]:
plt.xlabel('Epochs')
plt.ylabel('c_loss')
plt.plot(simclr_history.history['c_loss'], label='c_loss')
plt.plot(simclr_history.history['val_c_loss'], linestyle='dashed', label='val_c_loss')


plt.legend(loc='lower left')
plt.title('Evolution of contrastive loss')
plt.show()

In [ ]:
simclr_model.save_weights('simclr_small_window.weights.h5')
simclr_model.load_weights('simclr_full_bandwidth.weights.h5')

Training of a classification head on the test dataset

In [ ]:
# Freeze the encoder weights
simclr_model.encoder.trainable = False

classification_head_ft= keras.Sequential([
    layers.Input(shape=(n_embedding,)),
    layers.Dense(128, activation='relu'),
    layers.BatchNormalization(),
    layers.Dense(len(lb1)) #nb labels
], name='Classification_head')

# Create a new classification model
encoder_output = simclr_model.encoder(simclr_model.encoder.layers[0].input, training=False)
classification_model_cwru_pretrained = keras.Model(
    inputs=simclr_model.encoder.layers[0].input,
    outputs=classification_head_ft(encoder_output)
)

reduce_lr = ReduceLROnPlateau(
    monitor='val_loss', 
    factor=0.8,   
    patience=2          
)

classification_model_cwru_pretrained.compile(
    optimizer=keras.optimizers.Adam(3e-3),# 1e-2 
    loss=keras.losses.SparseCategoricalCrossentropy(from_logits=True), 
    metrics=['accuracy'])

classification_history = classification_model_cwru_pretrained.fit(
    ds_train_ft.repeat(), 
    epochs=100, 
    validation_data=ds_val_ft, 
    steps_per_epoch=int((0.1*ds_test_size) // batch_size),
    callbacks=[reduce_lr])

classification_model_cwru_pretrained.summary(show_trainable=True, expand_nested=True)

In [ ]:
plt.xlabel('Epochs')
plt.ylabel('loss')
plt.plot(classification_history.history['loss'], label='loss')
plt.plot(classification_history.history['val_loss'], linestyle='dashed', label='val_loss')


plt.legend(loc='lower left')
plt.title('Evolution of loss')
plt.show()

Test the effectiveness of the pretraining

In [ ]:
evaluation1 = classification_model_cwru_pretrained.evaluate(ds_test_ft)

print("Evaluation accuracy with pretraining on encoder: {:.2f}%".format(evaluation1[1]*100))

Test the accuracy with a random freeze encoder

In [ ]:
encoder = Encoder(input_shape,n_embedding,kernel_size)

simclr_model = SimCLRModel(encoder)

# Freeze the encoder weights
simclr_model.encoder.trainable = False

classification_head_ft= keras.Sequential([
    layers.Input(shape=(n_embedding,)),
    layers.Dense(128, activation='relu'),
    layers.BatchNormalization(),
    layers.Dense(len(lb1)) #nb labels
], name='Classification_head')

# Create a new classification model
encoder_output = simclr_model.encoder(simclr_model.encoder.layers[0].input, training=False)
simclr_model_not_pretrained = keras.Model(
    inputs=simclr_model.encoder.layers[0].input,
    outputs=classification_head_ft(encoder_output)
)

early_stopping = EarlyStopping(
    monitor='val_loss',  
    patience=3,       
    restore_best_weights=True  
)

reduce_lr = ReduceLROnPlateau(
    monitor='val_loss', 
    factor=0.1,         
    patience=2          
)

simclr_model_not_pretrained.compile(
    optimizer=keras.optimizers.Adam(), 
    loss=keras.losses.SparseCategoricalCrossentropy(from_logits=True), 
    metrics=['accuracy'])

classification_history = simclr_model_not_pretrained.fit(
    ds_train_ft.repeat(), 
    epochs=60, 
    validation_data=ds_val_ft, 
    steps_per_epoch=int((0.1*ds_test_size) // batch_size),
)

In [ ]:
evaluation1 = simclr_model_not_pretrained.evaluate(ds_test_ft)

print("Evaluation accuracy with random encoder: {:.2f}%".format(evaluation1[1]*100))